# 17. 值函数方法与 DQN 体系
值函数方法的核心思想是：与其直接学习“在每个状态下该怎么做”，不如先学习“每个状态或状态-动作对究竟值多少钱”。一旦价值估计足够准确，策略就可以通过贪心改进自动出现。

这条路线从表格型的 SARSA、Q-learning 出发，最终发展到深度 Q 网络（DQN）及其一系列稳定化变体，是离散动作空间强化学习最经典的方法谱系。

## 值函数控制与 DQN的形式化定义

            值函数方法试图估计最优动作价值函数
$$
Q^\star(s,a) = \mathbb{E}\left[r_t + \gamma \max_{a'} Q^\star(s_{t+1}, a') \mid s_t=s, a_t=a\right].
$$
一旦学到 $Q^\star$，就可以通过
$$
\pi^\star(s)=\arg\max_a Q^\star(s,a)
$$
导出最优策略。DQN 的创新在于用神经网络 $Q_\theta(s,a)$ 取代表格，从而把 TD 控制推广到高维感知空间。

## 输入、输出与参数化方式

            输入可以是离散状态、连续特征、图像帧或其他高维表征；输出是每个离散动作的价值估计。对离散动作空间，网络通常把同一状态下所有动作的 $Q$ 值一起输出：
$$
Q_\theta(s) = [Q_\theta(s,a_1), \dots, Q_\theta(s,a_{|\mathcal{A}|})].
$$
这使得动作选择只需做一次前向传播和一次 $\arg\max$。

## 结构分解与信息流

            DQN 的工程结构由四个关键部件组成：

1. 在线网络 $Q_\theta$：负责当前估计和动作选择；
2. 目标网络 $Q_{\bar{\theta}}$：生成较稳定的 bootstrap 目标；
3. 经验回放（Replay Buffer）：打破样本时间相关性；
4. ε-greedy 探索：在贪心利用之外保留随机探索。

Double DQN 进一步分离“选动作”和“评估动作”，缓解最大化带来的过估计；Dueling DQN 把状态价值和优势分开建模；PER 则优先抽取 TD 误差大的样本。

## 优化目标与训练机制

            DQN 通过最小化 TD 残差训练网络：
$$
\mathcal{L}(\theta) = \mathbb{E}_{(s,a,r,s')\sim \mathcal{D}}
\left[\left(Q_\theta(s,a) - y\right)^2\right],
$$
其中
$$
y = r + \gamma \max_{a'} Q_{\bar{\theta}}(s', a').
$$
Double DQN 把目标改写为
$$
y_{\text{double}} = r + \gamma Q_{\bar{\theta}}\left(s', \arg\max_{a'} Q_\theta(s', a')\right),
$$
以降低正偏差。

## 计算复杂度、统计性质与工程代价

            表格型值函数方法的更新开销很低，但只能处理小规模状态空间。DQN 把表示能力扩大到神经网络级别，却引入了目标漂移、样本相关性、过估计和训练不稳定等问题，因此必须依赖回放缓冲区、目标网络、梯度裁剪和奖励尺度控制。

从样本效率角度看，值函数方法通常优于纯 on-policy 策略梯度；但一旦动作空间连续化，基于 $\arg\max$ 的控制就不再方便，需要转向 Actor-Critic。

## 与相邻模型的差异

SARSA 是 on-policy 控制，它学习“当前行为策略真正会得到的价值”；Q-learning 是 off-policy 控制，它直接逼近最优贪心策略。DQN 继承了 Q-learning 的 off-policy 属性，并用深度网络扩展表示能力。Double / Dueling / PER 都是在 DQN 主框架上对偏差、方差和表示分解做工程修正。

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(13, 5.5))
ax.axis("off")
ax.set_xlim(0, 13)
ax.set_ylim(0, 5.5)

blocks = [
    (1.6, 2.8, 1.8, 1.0, "#4C78A8", "state s"),
    (4.0, 2.8, 2.2, 1.1, "#F58518", "online Q-net"),
    (7.1, 4.0, 2.4, 1.0, "#54A24B", "replay buffer"),
    (7.1, 1.6, 2.4, 1.0, "#E45756", "target Q-net"),
    (10.8, 2.8, 2.2, 1.2, "#72B7B2", "TD target\nr + gamma max Q_target"),
]
for x, y, w, h, color, text in blocks:
    rect = plt.Rectangle((x - w / 2, y - h / 2), w, h, facecolor=color, edgecolor="black")
    ax.add_patch(rect)
    ax.text(x, y, text, ha="center", va="center", fontsize=11)

ax.annotate("", xy=(2.9, 2.8), xytext=(2.5, 2.8), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.annotate("", xy=(5.3, 3.5), xytext=(5.9, 3.8), arrowprops=dict(arrowstyle="->", lw=1.6))
ax.annotate("", xy=(5.3, 2.1), xytext=(5.9, 1.8), arrowprops=dict(arrowstyle="->", lw=1.6))
ax.annotate("", xy=(9.6, 2.8), xytext=(8.3, 2.8), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(6.9, 5.0, "sample mini-batch", ha="center", fontsize=11)
ax.text(7.1, 0.55, "periodic / Polyak sync", ha="center", fontsize=11)
ax.set_title("Deep Q-Network with Replay Buffer and Target Network")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(12, 5))
ax.axis("off")
ax.set_xlim(0, 12)
ax.set_ylim(0, 5)

states = [(2.0, 2.5, "state s"), (6.0, 2.5, "next state s'"), (10.0, 2.5, "bootstrap target")]
for x, y, text in states:
    ax.add_patch(plt.Circle((x, y), 0.6, color="#4C78A8"))
    ax.text(x, y, text, ha="center", va="center", color="white", fontsize=11)

ax.annotate("", xy=(5.35, 2.5), xytext=(2.65, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(4.0, 2.85, "take action a", ha="center", fontsize=11)
ax.annotate("", xy=(9.35, 2.5), xytext=(6.65, 2.5), arrowprops=dict(arrowstyle="->", lw=1.8))
ax.text(8.0, 2.95, "r + gamma * V(s')", ha="center", fontsize=11)
ax.text(6.0, 1.1, "Bellman update: current estimate is corrected by a one-step target", ha="center", fontsize=12)
ax.set_title("Bellman Backup")
plt.show()

## 表格方法到深度方法的过渡

- **SARSA** 更新目标是当前策略实际选出的下一个动作：
  $$
  Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\left[r_t + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t,a_t)\right]
  $$
  因此更新更保守，也更能反映当前探索策略的真实风险。

- **Q-learning** 更新目标使用下一状态的最大动作价值：
  $$
  Q(s_t,a_t) \leftarrow Q(s_t,a_t) + \alpha\left[r_t + \gamma \max_a Q(s_{t+1}, a) - Q(s_t,a_t)\right]
  $$
  因而更偏向直接逼近最优策略。

- **DQN** 用神经网络近似 $Q$ 函数，把表格更新推广到高维观测空间，但必须用目标网络和回放缓冲区才能避免训练发散。

In [ ]:
# 兼容当前 Windows 环境：把临时目录固定到用户目录下的 ASCII 路径，
# 避免 scipy / sklearn 在中文工作目录下寻找临时文件时报错。
from pathlib import Path
import os
import warnings

temp_root = Path(os.environ.get("ML_DL_TMP", str(Path.home() / ".ml_dl_notebook_tmp")))
temp_root.mkdir(exist_ok=True)
os.environ["TMP"] = str(temp_root)
os.environ["TEMP"] = str(temp_root)

warnings.filterwarnings("ignore")

import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
random.seed(42)

sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]


import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)


print("临时目录:", temp_root)

In [ ]:
def run_cliff_control(method="q_learning", episodes=250, alpha=0.5, gamma=0.95, epsilon=0.1):
    rows, cols = 4, 6
    start = (3, 0)
    goal = (3, 5)
    cliff = {(3, c) for c in range(1, 5)}
    actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]
    q = np.zeros((rows, cols, len(actions)))
    returns = []

    def step(state, action_idx):
        dr, dc = actions[action_idx]
        nr = int(np.clip(state[0] + dr, 0, rows - 1))
        nc = int(np.clip(state[1] + dc, 0, cols - 1))
        next_state = (nr, nc)
        reward = -1.0
        done = False
        if next_state in cliff:
            reward = -100.0
            next_state = start
        elif next_state == goal:
            done = True
        return next_state, reward, done

    for _ in range(episodes):
        state = start
        action = np.random.randint(len(actions)) if np.random.rand() < epsilon else np.argmax(q[state])
        episode_return = 0.0
        for _ in range(200):
            next_state, reward, done = step(state, action)
            episode_return += reward
            if method == "sarsa":
                next_action = np.random.randint(len(actions)) if np.random.rand() < epsilon else np.argmax(q[next_state])
                target = reward if done else reward + gamma * q[next_state][next_action]
                q[state][action] += alpha * (target - q[state][action])
                state, action = next_state, next_action
            else:
                target = reward if done else reward + gamma * np.max(q[next_state])
                q[state][action] += alpha * (target - q[state][action])
                state = next_state
                action = np.random.randint(len(actions)) if np.random.rand() < epsilon else np.argmax(q[state])
            if done:
                break
        returns.append(episode_return)
    return q, np.array(returns)

q_sarsa, sarsa_returns = run_cliff_control(method="sarsa")
q_qlearning, qlearning_returns = run_cliff_control(method="q_learning")
cliff_df = pd.DataFrame(
    {
        "episode": np.arange(1, len(sarsa_returns) + 1),
        "SARSA": sarsa_returns,
        "Q-learning": qlearning_returns,
    }
)
cliff_df.head()

In [ ]:
cliff_long = cliff_df.melt(id_vars="episode", var_name="method", value_name="return")
cliff_long["smoothed_return"] = cliff_long.groupby("method")["return"].transform(
    lambda x: pd.Series(x).rolling(15, min_periods=1).mean()
)
plt.figure(figsize=(10, 5))
sns.lineplot(data=cliff_long, x="episode", y="smoothed_return", hue="method")
plt.title("SARSA vs Q-learning on Cliff Walking")
plt.ylabel("moving average return")
plt.show()

In [ ]:
from collections import deque

class LineWorldEnv:
    def __init__(self, n_states=7, max_steps=14):
        self.n_states = n_states
        self.max_steps = max_steps
        self.reset()

    def _obs(self):
        obs = np.zeros(self.n_states, dtype=np.float32)
        obs[self.pos] = 1.0
        return obs

    def reset(self):
        self.pos = np.random.randint(0, self.n_states - 2)
        self.steps = 0
        return self._obs()

    def step(self, action):
        self.steps += 1
        self.pos = int(np.clip(self.pos + (1 if action == 1 else -1), 0, self.n_states - 1))
        reward = 1.0 if self.pos == self.n_states - 1 else -0.03
        done = self.pos == self.n_states - 1 or self.steps >= self.max_steps
        return self._obs(), reward, done

class QNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, output_dim),
        )

    def forward(self, x):
        return self.net(x)

env = LineWorldEnv()
q_net = QNet(env.n_states, 2)
target_net = QNet(env.n_states, 2)
target_net.load_state_dict(q_net.state_dict())
optimizer = torch.optim.Adam(q_net.parameters(), lr=1e-3)
replay = deque(maxlen=3000)
gamma = 0.95
epsilon = 1.0
batch_size = 64
dqn_returns = []

for episode in range(220):
    state = env.reset()
    done = False
    episode_return = 0.0
    while not done:
        state_tensor = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        if np.random.rand() < epsilon:
            action = np.random.randint(2)
        else:
            with torch.no_grad():
                action = int(q_net(state_tensor).argmax(dim=1).item())

        next_state, reward, done = env.step(action)
        replay.append((state, action, reward, next_state, float(done)))
        state = next_state
        episode_return += reward

        if len(replay) >= batch_size:
            batch = random.sample(replay, batch_size)
            states_b = torch.tensor(np.array([b[0] for b in batch]), dtype=torch.float32)
            actions_b = torch.tensor([b[1] for b in batch], dtype=torch.long).unsqueeze(1)
            rewards_b = torch.tensor([b[2] for b in batch], dtype=torch.float32)
            next_states_b = torch.tensor(np.array([b[3] for b in batch]), dtype=torch.float32)
            dones_b = torch.tensor([b[4] for b in batch], dtype=torch.float32)

            q_values = q_net(states_b).gather(1, actions_b).squeeze(1)
            with torch.no_grad():
                next_q = target_net(next_states_b).max(dim=1).values
                targets = rewards_b + gamma * next_q * (1.0 - dones_b)

            loss = nn.functional.mse_loss(q_values, targets)
            optimizer.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(q_net.parameters(), 1.0)
            optimizer.step()

    epsilon = max(0.05, epsilon * 0.985)
    if (episode + 1) % 15 == 0:
        target_net.load_state_dict(q_net.state_dict())
    dqn_returns.append(episode_return)

dqn_df = pd.DataFrame(
    {
        "episode": np.arange(1, len(dqn_returns) + 1),
        "return": dqn_returns,
        "moving_avg": pd.Series(dqn_returns).rolling(15, min_periods=1).mean(),
    }
)
dqn_df.tail()

In [ ]:
plt.figure(figsize=(10, 5))
sns.lineplot(data=dqn_df, x="episode", y="moving_avg")
plt.title("DQN Learning Curve on a Tiny LineWorld")
plt.ylabel("moving average return")
plt.show()

with torch.no_grad():
    state_grid = torch.eye(env.n_states)
    q_grid = q_net(state_grid).numpy()
q_value_df = pd.DataFrame(q_grid, columns=["go_left", "go_right"])
q_value_df["state"] = np.arange(env.n_states)
q_value_df

In [ ]:
q_long = q_value_df.melt(id_vars="state", var_name="action", value_name="q_value")
plt.figure(figsize=(10, 5))
sns.barplot(data=q_long, x="state", y="q_value", hue="action")
plt.title("Q-values Learned by DQN")
plt.show()

In [ ]:
true_q = np.array([1.0, 0.8, 0.6])
single_estimates = []
double_estimates = []
for _ in range(10000):
    q1 = true_q + np.random.normal(0, 0.5, size=3)
    q2 = true_q + np.random.normal(0, 0.5, size=3)
    single_estimates.append(np.max(q1))
    double_estimates.append(q2[np.argmax(q1)])

bias_df = pd.DataFrame(
    {
        "estimator": ["single max", "double estimate", "true max"],
        "value": [np.mean(single_estimates), np.mean(double_estimates), np.max(true_q)],
    }
)

td_errors = np.linspace(0.05, 2.0, 12)
priorities = (td_errors + 1e-3) ** 0.6
priorities = priorities / priorities.sum()
per_df = pd.DataFrame({"td_error": td_errors, "sampling_prob": priorities})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sns.barplot(data=bias_df, x="estimator", y="value", palette="viridis", ax=axes[0])
axes[0].set_title("Why Double DQN Reduces Overestimation")
sns.barplot(data=per_df, x="td_error", y="sampling_prob", color="#F58518", ax=axes[1])
axes[1].set_title("Prioritized Replay Sampling Probability")
axes[1].tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 使用建议与失效模式

- 如果状态空间较小，优先先用表格型 SARSA / Q-learning 把更新逻辑理解清楚，再进入 DQN。
- DQN 适合**离散动作空间**。一旦动作连续化，$\max_a Q(s,a)$ 的求解会变得困难，应转向 DDPG、TD3、SAC 等 Actor-Critic 方法。
- 训练发散通常不是网络“太弱”，而是 TD 目标漂移过快、奖励尺度失衡、探索不足或回放缓冲区分布质量太差。
- Double DQN、Dueling DQN、PER 不应被理解成彼此竞争的独立算法，而应理解为对 DQN 主体框架不同脆弱点的修补。

## 主要资料

- DQN（Nature 2015）: https://www.nature.com/articles/nature14236
- Prioritized Experience Replay（2015）: https://arxiv.org/abs/1511.05952
- Dueling Network Architectures（2016）: https://arxiv.org/abs/1511.06581